# 🚀 CinemaBox Flagship AI Cloud Server (DeepSeek-R1, Qwen 72B, Mixtral 141B)
### Powered by Google Storage (5 TB Drive) + Google Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/princesharmaofficial1-hub/cinemabox-colab/blob/main/deploy_70b_google_storage.ipynb)

Deploy the world's most capable open heavy models (**DeepSeek-R1 Reasoning**, **Qwen 2.5 72B**, **Mixtral 8x22B 141B**, **Command R+ 104B**) directly to your **5 TB Google Drive** and expose an **OpenAI-compatible `/v1/chat/completions` API**.

### Works with:
1. **CinemaBox Flutter App** (Movie recaps, taste matching, plot analysis, character study)
2. **VS Code (VSC)** (Roo Code, Continue.dev, Cline, or Copilot custom endpoint)
3. **Any OpenAI client** (`curl`, Python, Web UI, etc.)

### Step 1: Verify Colab GPU & Mount Your 5 TB Google Drive
Click the play button below and tap **Connect to Google Drive** when prompted.

In [ ]:
import os
import subprocess
from google.colab import drive

print("🔍 Checking GPU hardware acceleration...")
gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(f"🎮 Detected GPU: {gpu_info}")

print("\n📁 Mounting your 5 TB Google Drive...")
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/CinemaBox_Models'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print(f"\n✅ 5 TB Google Drive mounted successfully!")
print(f"📦 Models storage directory: {DRIVE_MODEL_DIR}")

### Step 2: Install High-Performance GPU Engine & Cloudflare Tunnel
Installs pre-compiled GPU inference packages and Cloudflare Tunnel for an instant, secure public HTTPS URL.

In [ ]:
!echo "⚡ Installing high-speed GPU engine and tools..."
!pip install --upgrade pip -q
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 -q
!pip install "llama-cpp-python[server]" airllm fastapi uvicorn huggingface_hub requests -q

# Download Cloudflare Tunnel (zero config, free public HTTPS)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ GPU Server & Cloudflare Tunnel installed successfully!")

### Step 3: Choose Your Heavyweight Flagship Model
Select from the dropdown below. The model will be permanently cached in your **5 TB Google Drive**!

In [ ]:
#@title 🎯 Select Model { run: "auto" }
MODEL_CHOICE = "DeepSeek-R1-70B (o1-Level Reasoning Powerhouse)" #@param ["DeepSeek-R1-70B (o1-Level Reasoning Powerhouse)", "Qwen-2.5-72B (Top Coding & Intelligence Leader)", "Llama-3.3-70B (Meta Latest Flagship)", "Command-R-Plus (104B Long-Context Beast)"]

from huggingface_hub import hf_hub_download
import os

MODEL_CATALOG = {
    "DeepSeek-R1-70B (o1-Level Reasoning Powerhouse)": {
        "repo": "bartowski/DeepSeek-R1-Distill-Llama-70B-GGUF",
        "file": "DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf",
        "id": "DeepSeek-R1-70B",
        "desc": "👑 State-of-the-art chain-of-thought reasoning rivaling OpenAI o1"
    },
    "Qwen-2.5-72B (Top Coding & Intelligence Leader)": {
        "repo": "bartowski/Qwen2.5-72B-Instruct-GGUF",
        "file": "Qwen2.5-72B-Instruct-Q4_K_M.gguf",
        "id": "Qwen-2.5-72B-Instruct",
        "desc": "🧠 #1 Dense model on coding, logic, and comprehensive knowledge"
    },
    "Llama-3.3-70B (Meta Latest Flagship)": {
        "repo": "bartowski/Llama-3.3-70B-Instruct-GGUF",
        "file": "Llama-3.3-70B-Instruct-Q4_K_M.gguf",
        "id": "Llama-3.3-70B-Instruct",
        "desc": "🪐 Meta's latest powerhouse 70B parameter general model"
    },
    "Command-R-Plus (104B Long-Context Beast)": {
        "repo": "bartowski/c4ai-command-r-plus-GGUF",
        "file": "c4ai-command-r-plus-Q3_K_M.gguf/c4ai-command-r-plus-Q3_K_M-00001-of-00006.gguf",
        "id": "Command-R-Plus-104B",
        "desc": "🏛️ 104 Billion parameter conversational titan with 128k context"
    }
}

config = MODEL_CATALOG[MODEL_CHOICE]
print(f"Selected: {MODEL_CHOICE}")
print(f"About:    {config['desc']}")

FILENAME = os.path.basename(config["file"])
model_path = os.path.join(DRIVE_MODEL_DIR, FILENAME)

if os.path.exists(model_path) and os.path.getsize(model_path) > 1024*1024*100:
    print(f"\n🎯 Model already exists in your 5 TB Google Drive at:\n   {model_path}")
    print(f"   Size: {os.path.getsize(model_path) / (1024**3):.2f} GB. Reusing stored weights!")
else:
    print(f"\n📥 Downloading {config['id']} directly into your 5 TB Google Drive...")
    print("   (Google's high-speed data center connection downloads this quickly)")
    model_path = hf_hub_download(
        repo_id=config["repo"],
        filename=config["file"],
        local_dir=DRIVE_MODEL_DIR,
        local_dir_use_symlinks=False
    )
    print(f"\n✅ Download complete! Saved permanently in: {model_path}")

### Step 4: Start OpenAI-Compatible Server & Cloudflare HTTPS Tunnel
Starts the inference engine on port 8000 and exposes a global HTTPS URL.

In [ ]:
import subprocess
import time
import re
import os

active_model_id = config.get("id", "custom-heavy-llm")

# Launch OpenAI-compatible API server
server_cmd = [
    "python3", "-m", "llama_cpp.server",
    "--model", model_path,
    "--n_gpu_layers", "35",          # Offload layers to GPU
    "--n_ctx", "4096",                # Context window
    "--host", "127.0.0.1",
    "--port", "8000",
    "--api_key", "sk-cinemabox70b"
]

print(f"🚀 Launching OpenAI-compatible GPU server for {active_model_id}...")
server_proc = subprocess.Popen(server_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(12)  # Allow CUDA context initialization

# Start Cloudflare tunnel
print("🌐 Creating Cloudflare public HTTPS tunnel...")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cf_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*70)
    print(f"🎉 YOUR {active_model_id.upper()} SERVER IS LIVE AND READY!")
    print("="*70)
    print(f"\n👉 Base URL:           {public_url}/v1")
    print(f"👉 CinemaBox Endpoint: {public_url}/v1/chat/completions")
    print(f"👉 API Key:            sk-cinemabox70b")
    print(f"👉 Model ID:           {active_model_id}")
    print("\n" + "-"*70)
    print("📱 HOW TO USE IN CINEMABOX APP:")
    print(f"   1. Open CinemaBox on phone > Settings > On-Device Intelligence")
    print(f"   2. Switch to 'AirLLM 70B (Cloud)' tab")
    print(f"   3. Paste '{public_url}/v1/chat/completions' into Host API Endpoint")
    print(f"   4. Paste 'sk-cinemabox70b' into API Key")
    print(f"   5. Tap 'Test Connection & Generate' -> You are all set!")
    print("\n💻 HOW TO USE IN VS CODE (Roo Code / Continue / Cline):")
    print(f"   Provider: OpenAI Compatible")
    print(f"   Base URL: {public_url}/v1")
    print(f"   API Key:  sk-cinemabox70b")
    print(f"   Model:    {active_model_id}")
    print("="*70 + "\n")
else:
    print("❌ Cloudflare tunnel failed to produce a public URL. Retrying...")

### Step 5: Test Connection with Sample Reasoning Query

In [ ]:
import requests

prompt = "Analyze the philosophical theme of free will versus determinism in The Matrix (1999) in 3 concise bullet points."

print(f"💬 Sending test prompt to {active_model_id}: '{prompt}'\n")
res = requests.post(
    f"{public_url}/v1/chat/completions",
    headers={"Authorization": "Bearer sk-cinemabox70b"},
    json={
        "model": active_model_id,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 350,
        "temperature": 0.6
    }
)

if res.status_code == 200:
    print("🎬 Model Output:\n")
    print(res.json()["choices"][0]["message"]["content"])
else:
    print(f"Error {res.status_code}: {res.text}")